# Model 1 - Donor Lapse Risk (Leakage-Safe)

Goal: predict whether a donor will lapse (no donation) in the next 90 days.

Leakage guards:
- Features use only donation history up to snapshot date.
- Label uses donations after snapshot date only.
- Chronological train/test split.


In [1]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.inspection import permutation_importance
import joblib

DATA_DIR = Path('.')
ARTIFACTS_DIR = DATA_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

donations = pd.read_csv(DATA_DIR / 'donations.csv', parse_dates=['donation_date'])
supporters = pd.read_csv(DATA_DIR / 'supporters.csv', parse_dates=['created_at', 'first_donation_date'])

def time_split(df, time_col, frac=0.8):
    df = df.sort_values(time_col).copy()
    cut = int(len(df) * frac)
    return df.iloc[:cut].copy(), df.iloc[cut:].copy()

def build_preprocessor(X):
    num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat_cols = [c for c in X.columns if c not in num_cols]
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('pow', PowerTransformer(method='yeo-johnson', standardize=False)), ('sc', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ])


FileNotFoundError: [Errno 2] No such file or directory: 'donations.csv'

In [ ]:
rows = []
d = donations.sort_values(['supporter_id', 'donation_date']).copy()
for sid, g in d.groupby('supporter_id'):
    g = g.reset_index(drop=True)
    for i in range(1, len(g)-1):
        snapshot_date = g.loc[i, 'donation_date']
        hist = g.loc[:i]           # only known at snapshot
        future = g.loc[i+1:]       # only for label
        has_future_90 = ((future['donation_date'] > snapshot_date) & (future['donation_date'] <= snapshot_date + pd.Timedelta(days=90))).any()
        rows.append({
            'supporter_id': sid,
            'snapshot_date': snapshot_date,
            'lapse_90d': int(not has_future_90),
            'days_since_last_donation': (snapshot_date - hist['donation_date'].max()).days,
            'donation_count_hist': len(hist),
            'sum_estimated_value_hist': hist['estimated_value'].fillna(0).sum(),
            'avg_estimated_value_hist': hist['estimated_value'].fillna(0).mean(),
            'recurring_rate_hist': hist['is_recurring'].astype(int).mean(),
            'monetary_share_hist': (hist['donation_type'] == 'Monetary').mean(),
            'social_channel_share_hist': (hist['channel_source'] == 'SocialMedia').mean(),
        })

m1 = pd.DataFrame(rows).merge(
    supporters[['supporter_id', 'supporter_type', 'relationship_type', 'region', 'country', 'acquisition_channel']],
    on='supporter_id', how='left'
)

train_df, test_df = time_split(m1, 'snapshot_date', 0.8)
X_train = train_df.drop(columns=['lapse_90d', 'snapshot_date'])
y_train = train_df['lapse_90d']
X_test = test_df.drop(columns=['lapse_90d', 'snapshot_date'])
y_test = test_df['lapse_90d']

pre = build_preprocessor(X_train)
predictive = Pipeline([('pre', pre), ('model', RandomForestClassifier(n_estimators=300, min_samples_leaf=4, random_state=42))])
explanatory = Pipeline([('pre', pre), ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))])

predictive.fit(X_train, y_train)
explanatory.fit(X_train, y_train)

prob = predictive.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)
print({'roc_auc': roc_auc_score(y_test, prob), 'avg_precision': average_precision_score(y_test, prob), 'f1': f1_score(y_test, pred)})

imp = permutation_importance(predictive, X_test, y_test, n_repeats=8, random_state=42)
print(pd.DataFrame({'feature': X_test.columns, 'importance': imp.importances_mean}).sort_values('importance', ascending=False).head(10))

joblib.dump(predictive, ARTIFACTS_DIR / 'model1_predictive.joblib')
joblib.dump(explanatory, ARTIFACTS_DIR / 'model1_explanatory.joblib')


{'roc_auc': 0.5881410256410257, 'avg_precision': 0.257182644543686, 'f1': 0.34146341463414637}
                      feature    importance
4    avg_estimated_value_hist  1.024590e-02
6         monetary_share_hist  1.024590e-02
11                    country  4.098361e-03
3    sum_estimated_value_hist  4.098361e-03
1    days_since_last_donation  0.000000e+00
12        acquisition_channel -2.775558e-17
2         donation_count_hist -2.049180e-03
7   social_channel_share_hist -2.049180e-03
9           relationship_type -4.098361e-03
10                     region -6.147541e-03


['artifacts\\model1_explanatory.joblib']

In [ ]:
# Final business insights block (human-readable + actionable)

print('\n=== BUSINESS TAKEAWAYS: MODEL 1 (DONOR LAPSE RISK) ===')

scored = m1.copy()
features = [c for c in scored.columns if c not in ['lapse_90d', 'snapshot_date']]
scored['lapse_risk_score'] = predictive.predict_proba(scored[features])[:, 1]

# Risk bands for actioning
scored['risk_band'] = pd.cut(scored['lapse_risk_score'], bins=[-0.001, 0.35, 0.65, 1.0], labels=['Low', 'Medium', 'High'])

print('Risk band distribution:')
display(scored['risk_band'].value_counts(dropna=False).rename_axis('risk_band').reset_index(name='count'))

# Highest-risk donors for immediate follow-up
top_risk = scored.sort_values('lapse_risk_score', ascending=False).head(20)[
    ['supporter_id', 'lapse_risk_score', 'supporter_type', 'acquisition_channel', 'days_since_last_donation', 'donation_count_hist', 'avg_estimated_value_hist']
]
print('\nTop 20 at-risk donors (worklist):')
display(top_risk)

# Segment-level insights
segment_risk = scored.groupby(['supporter_type', 'acquisition_channel'], dropna=False)['lapse_risk_score'].mean().reset_index().sort_values('lapse_risk_score', ascending=False)
print('\nHighest-risk donor segments:')
display(segment_risk.head(10))

print('\nActionable guidance:')
print('- Start weekly retention outreach with the Top 20 list.')
print('- Use high-risk segments table to tailor messaging and cadence by channel/type.')
print('- Prioritize donors with high risk + historically high value for fastest impact.')
print('- This is a predictive triage signal; final action should be staff-reviewed.')